In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
import os
from functools import partial

import astropy.units as u
import matplotlib.pyplot as plt  # type: ignore
import numpy as np
from joblib import Parallel, delayed
from pixell import curvedsky
from tqdm.auto import tqdm

from notebooks.ksw_joblib import KSW_joblib
from scripts import core  # note the small c to import the core module and not the class

from scripts import Core
from scripts.utils import remove_mono_dipole, setup_logging
from scripts.utils.plots import plot_cl_alm, plot_patches

# Monkey patch the core module
core.KSW = KSW_joblib


In [ ]:
logger = setup_logging(__name__, level=logging.DEBUG)

## Setup

In [ ]:
core = Core(
    [
        "settings/planck.json",
        "--nsims",
        "1",
        "--narray",
        "1",
        # "--polarizations",
        # "T",
        # "E",
    ]
)
core.is_main_job = True  # fix an issue with missing slurm job array

## Alm

In [ ]:
from scripts.almgen import generate_alm_ng

This code generates the alms

$$a_{\ell m} = a_{\ell m}^{{G}} + f_{NL}^X a_{\ell m}^{NG}$$
with
$$a_{\ell m}^{NG,loc'} = \int dr r^2 \left[ \alpha_\ell(r)\left(\int d^2 \hat{n} Y_{\ell m}^\star (\hat{n}) B(r,\hat{n})^2 \right)\right]$$
and
$$\alpha_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^2 \Delta_\ell^T(k) j_\ell(k r)$$
$$\beta_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^{-1} \Delta_\phi \Delta_\ell^T(k) j_\ell(k r)$$
$$B(r, \hat{n}) = \sum_{\ell,m} \frac{\beta_\ell (r)}{C_\ell} a_{\ell m} Y_{\ell m}$$
where $\Delta_\phi$ is primordial normalization, $\Delta_\ell^T(k)$ is the transfer function, $j_\ell(k r)$ are the spherical bessel functions   

In [ ]:
alm_l = np.array(
    [core.data.compute_alm_sim(core.lensing) for _ in range(core.nsims)],
    dtype=core.c_dtype,
)

plot_cl_alm(
    alm_l[0, 0],
    title="alm_l",
    plot_camb=True,
    c_ells=core.c_ells,
    camb_noise=True,
    noise=core.noise_ell[0],
    beam_width=core.beam_width,
)

fnls = core.rng.uniform(core.fnl_min, core.fnl_max, core.fnl_shape)

alm_ng = generate_alm_ng(core, alm_l)

plot_cl_alm(alm_ng[0, 0], title="alm_ng")

alms = alm_l + fnls[:, None, :] * alm_ng
alms = remove_mono_dipole(alms)

plot_cl_alm(
    alms[0, 0],
    title="alms final",
    plot_camb=True,
    c_ells=core.c_ells,
    camb_noise=True,
    noise=core.noise_ell,
    beam_width=core.beam_width,
)

In [ ]:
for pol in range(core.npol):
    logger.info(alms.shape)
    ells = np.arange(2, core.nell)
    scale = ells * (ells + 1) / 2 / np.pi
    N_ell = core.noise_ell[pol, 2:]
    cl_cmb = core.c_ells[2 : core.nell, pol]
    cl_sim = curvedsky.alm2cl(alms[0, pol])[2 : core.nell]
    beam = core.beam_ell[pol, 2:]

    plot_func = plt.semilogy
    plot_func(scale * cl_sim, label=r"sim", linestyle=":")
    plot_func(scale * N_ell, label=r"$N_\ell$")
    plot_func(scale * cl_cmb, label=r"$C_{\ell}^{CMB}$")
    plot_func(scale * (cl_cmb * beam**2 + N_ell), label=r"$C_\ell^{CMB} + N_\ell$")
    plt.legend()
    plt.xlabel("Multipole moment (l)")
    plt.ylabel(r"$\ell(\ell+1)/2\pi\;C_{\ell}$")
    plt.grid(True)
    # plt.ylim([3e2, 4e4])
    noise_muk2arcmin = (core.noise_scale_tt * u.radian).to_value(u.arcmin)
    plt.title(
        f"Pol: {pol}, Noise [$\\mu K$ arcmin]: {noise_muk2arcmin:.4f}, Beam Width [rad]: {core.beam_width:.4f}"
    )
    plt.show()

## PatchGen

In [ ]:
from scripts.patchgen import cutSqPatches_lenspyx, cutSqPatches_pixell, get_fs_patch_geo

In [ ]:
# Here we get the geometry of our patches in a tuple
patch_geo = get_fs_patch_geo(core)

# we also setup the cutPatches function to use either pixell or lenspyx
# depending on if we are doing lensing or not, and provide a lot of
# arguments that are needed and will stay constant
# we cannot abuse the python scope here since these will need to be pickled
common_settings = [
    core.lmax,
    core.plot_dir,
    core.base_name,
    core.npatches,
    core.c_ells,
    *patch_geo,
]
if core.lensing:
    max_l = core.cosmo_params["max_l"]
    cl_phi = core.cosmo._camb_data.get_lens_potential_cls(  # type: ignore
        max_l, CMB_unit="muK", raw_cl=True
    )[
        :, 0
    ]  # what do to about the pol here???!?!?!?!?

    cutPatches = partial(
        cutSqPatches_lenspyx, *common_settings, max_l, cl_phi, core.nside, core.r_dtype
    )
else:
    cutPatches = partial(cutSqPatches_pixell, *common_settings)

## Start the patch generation
# create the array to store the patches
patches = np.empty(core.patch_shape, dtype=core.r_dtype)

# We use joblib.parallel to generate the patches in parallel
# by default (temp_folder=None) this will use a ram disk /dev/shm
# if the data files are larger than the available memory, about 1TB, it will error
# so we give it a temp folder to use, which wont have that problem
temp_folder = os.environ.get("SCRATCH", None)
logger.debug(f"Using temp folder for patch generation: {temp_folder}")
patch_generator = Parallel(
    n_jobs=-1,
    return_as="generator",
    temp_folder=temp_folder,
)(delayed(cutPatches)(alms[i, j], fnls[i, j], False) for i, j in core.sim_pol)

# Get our data from the generator, only update logging every 100 runs, takes a long time
for idx, result in enumerate(
    tqdm(patch_generator, desc="patch progress", total=core.sim_pol_len)
):
    i, pol = core.sim_pol[idx]
    patches[i, pol] = result

In [ ]:
for pol in range(core.npol):
    plot_patches(patches[0, pol], 10, title=f"Pol {pol}")

## Estimator

In [ ]:
def alm_step_loader(idx):
    logger.info("Running KSW step %s", idx)
    return core.data.compute_alm_sim(core.lensing)

theta_batch = max(512, int(np.floor(1.5 * core.lmax + 1)))
core.ksw.step_batch(alm_step_loader, range(100), theta_batch=theta_batch)

In [ ]:
fisher = float(core.ksw.compute_fisher())
logger.info("Fisher: %s, standard deviation: %s", fisher, np.sqrt(1 / fisher))

In [ ]:
def alm_loader(i):
    logger.info("Loading alm %s, FNL %s", i, fnls[i])
    return alms[i]


idxs = range(alms.shape[0])
estimates = core.ksw.compute_estimate_batch(
    alm_loader, idxs, theta_batch=theta_batch, fisher=fisher
)